In [ ]:
# need to install libraries for reading csv file/calling google api
##!pip install --upgrade google-maps-services-python
#!pip install pandas requests 


In [ ]:
import pandas as pd
import requests
import math
import os

In [ ]:
import os
import googlemaps
#from dotenv import load_dotenv
from typing import List, Union, Optional

#securely handle API key by setting it as an environment variable
# load_dotenv(dotenv_path=".env")
# print(os.environ)
# API_KEY = os.getenv("GOOGLE_API_KEY")

API_KEY = "my api key hehe"  # hardcode just to check
gmaps = googlemaps.Client(key=API_KEY)


#check if the key is set properly
# if not API_KEY:
#     raise ValueError("Please set the GOOGLE_MAPS_API_KEY environment variable.")
print("API key loaded:", API_KEY is not None)

print(type(gmaps))
print(dir(gmaps)) 

# #connect to google maps using my API key
# gmaps = googlemaps.Client(key=API_KEY)

In [ ]:
# csv_path = "classes_cleaned.csv" #this path needs to be updated!!!!
# df = pd.read_csv(csv_path)
# print(df.head())

In [ ]:
#this function takes two locations and return walking time between them 
def get_walking_time(origin, destination):
    try:
        response = gmaps.distance_matrix( #type: ignore
            
            origins=[origin],
            destinations=[destination],
            mode="walking",
            departure_time="now"
        )

        #extract walking time from the response
        info = response['rows'][0]['elements'][0]
        if info['status'] == 'OK':
            seconds = info['duration']['value'] #get time in second
            minutes = seconds / 60 #convert to minutes
            return round(minutes, 2) #round to 2dp
        else :
            return None
    except Exception as e:
        print("Error while calculating walking time from {origin} to {destination}: {e}")
        return None

#print(get_walking_time("hendrick House, IL", "Illinois Street Residence Halls, IL"))

In [ ]:
def compute_travel_satsifaction_score(schedule, max_late_minutes, df, api_key):
    satisfactory = 0
    considered = 0

    #loop through each pair
    for i in range(len(schedule) - 1):
        class1 = schedule[i]
        class2 = schedule[i + 1]
        
        #compute time gap between in mins
        t1_end_time = int(class1["end"].split(":")[0]) * 60 + int(class1["end"].split(":")[1])
        t2_start_time = int(class2["start"].split(":")[0]) * 60 + int(class2["end"].split(":")[1])
        gap = t2_start_time - t1_end_time

        #if gap too long, not worried about travel 
        if gap > 30:
            continue

        
        #calculate travel time using google 
        travel_minutes = get_walking_time(class1["building"], class2["building"])
        if travel_minutes is None:
            print("Could not get travel time")
            continue

        considered += 1

        #check if travel time fits user's acceptable limit
        if travel_minutes <= 10 + max_late_minutes:
            satisfactory += 1


    #calculate and return percentage 
    if considered == 0:
        return None
    else:
        return satisfactory / considered * 100 

In [ ]:
#geocode returns a python list of one or more JSON objects 
def get_coordinates(location):
    try: 
        geocode_result = gmaps.geocode(location) #type:ignore
        #check 1.api actually retruned something 2.first result contains geometry key
        if geocode_result and 'geometry' in geocode_result[0]:
            #first result ->geometry dictionary -> location dictionary -> extract lat and lng
            lat = geocode_result[0]['geometry']['location']['lat']
            lng = geocode_result[0]['geometry']['location']['lng']
            return (lat, lng)
        else:
            return None
        
    except Exception as e:
        print("Error while getting coordinates for {location} : {e}")
        return None

In [ ]:
#this function takes two locations and return walking time between them 
def get_distance_meters(origin, destination):
    try:
        response = gmaps.distance_matrix( #type: ignore
            
            origins=[origin],
            destinations=[destination],
            mode="walking",
            units="metric",
        )

        
        info = response['rows'][0]['elements'][0]
        if info['status'] == 'OK':
            return info['distance']['value'] #distance in meters
        else :
            return None
        
    except Exception as e:
        print("Error while finding distance from the {origin} to {destinations}")
        return None
        

In [ ]:
def compute_proximity_score(schedule, center, radius):
    center_coords = get_coordinates(center)
    if center_coords is None:
        print("Error: Could not find coordinates for {center}")
        return 0.0

    center_lat, center_lng = center_coords
    num_classes = len(schedule)
    
    in_range = 0
    
    #loop through each location in this schedule 
    for location in schedule:
        if not location:
            continue

        class_coords = get_coordinates(location)
        if class_coords is None:
            print("Could not find the coordinate for {loc} for this is skipped")
            continue
        
        class_lat, class_lng = class_coords

        distance = 2 #need a distance calculation function

        if distance <= radius: 
            in_range += 1

    percentage = (in_range / num_classes) * 100

    return round(percentage, 2)
    

